In [ ]:
#imports
import os
import glob
import random
import time
import numpy as np
import pandas as pd
import librosa
import yt_dlp
from typing import Optional


Genres and subgenres in part 2

Classes: 12 — ['Classical/Instrumental', 'Country/Folk', 'Electronic', 'Hip-Hop/R&B', 'House/Dance', 'Jazz/Blues', 'Latin', 'Metal', 'Pop', 'Reggae', 'Rock', 'World/Other']

In [18]:
# map every subgenre to a parent genre

GENRE_MAP = {
    # Rock
    'rock': 'Rock', 'alt-rock': 'Rock', 'punk-rock': 'Rock',
    'hard-rock': 'Rock', 'psych-rock': 'Rock', 'grunge': 'Rock',
    'rock-n-roll': 'Rock', 'emo': 'Rock', 'indie': 'Rock',
    'alternative': 'Rock', 'rockabilly': 'Rock',

    # Metal
    'metal': 'Metal', 'heavy-metal': 'Metal', 'death-metal': 'Metal',
    'black-metal': 'Metal', 'metalcore': 'Metal', 'grindcore': 'Metal',
    'hardcore': 'Metal', 'goth': 'Metal', 'industrial': 'Metal',

    # Electronic
    'edm': 'Electronic', 'electronic': 'Electronic', 'techno': 'Electronic',
    'detroit-techno': 'Electronic', 'minimal-techno': 'Electronic',
    'trance': 'Electronic', 'dubstep': 'Electronic', 'drum-and-bass': 'Electronic',
    'idm': 'Electronic', 'electro': 'Electronic', 'breakbeat': 'Electronic',
    'hardstyle': 'Electronic', 'ambient': 'Electronic',

    # House/Dance
    'house': 'House/Dance', 'deep-house': 'House/Dance',
    'chicago-house': 'House/Dance', 'progressive-house': 'House/Dance',
    'dance': 'House/Dance', 'disco': 'House/Dance', 'club': 'House/Dance',
    'garage': 'House/Dance', 'dancehall': 'House/Dance',

    # Pop
    'pop': 'Pop', 'indie-pop': 'Pop', 'synth-pop': 'Pop',
    'power-pop': 'Pop', 'k-pop': 'Pop', 'j-pop': 'Pop',
    'cantopop': 'Pop', 'mandopop': 'Pop', 'j-idol': 'Pop',
    'j-dance': 'Pop', 'party': 'Pop', 'happy': 'Pop',
    'pop-film': 'Pop', 'disney': 'Pop', 'show-tunes': 'Pop',

    # Hip-Hop / R&B
    'hip-hop': 'Hip-Hop/R&B', 'r-n-b': 'Hip-Hop/R&B', 'soul': 'Hip-Hop/R&B',
    'funk': 'Hip-Hop/R&B', 'groove': 'Hip-Hop/R&B', 'trip-hop': 'Hip-Hop/R&B',

    # Latin
    'latin': 'Latin', 'latino': 'Latin', 'salsa': 'Latin',
    'samba': 'Latin', 'reggaeton': 'Latin', 'tango': 'Latin',
    'forro': 'Latin', 'pagode': 'Latin', 'sertanejo': 'Latin',
    'mpb': 'Latin', 'brazil': 'Latin', 'bossanova': 'Latin',
    'romance': 'Latin', 'spanish': 'Latin',

    # Jazz / Blues
    'jazz': 'Jazz/Blues', 'blues': 'Jazz/Blues', 'gospel': 'Jazz/Blues',
    'soul': 'Jazz/Blues',

    # Classical / Instrumental
    'classical': 'Classical/Instrumental', 'opera': 'Classical/Instrumental',
    'piano': 'Classical/Instrumental', 'guitar': 'Classical/Instrumental',
    'new-age': 'Classical/Instrumental', 'sleep': 'Classical/Instrumental',
    'study': 'Classical/Instrumental',

    # Country / Folk
    'country': 'Country/Folk', 'folk': 'Country/Folk', 'bluegrass': 'Country/Folk',
    'honky-tonk': 'Country/Folk', 'singer-songwriter': 'Country/Folk',
    'songwriter': 'Country/Folk', 'acoustic': 'Country/Folk',

    # Reggae
    'reggae': 'Reggae', 'dub': 'Reggae', 'ska': 'Reggae',

    # World / Other
    'world-music': 'World/Other', 'afrobeat': 'World/Other',
    'indian': 'World/Other', 'iranian': 'World/Other',
    'turkish': 'World/Other', 'malay': 'World/Other',
    'french': 'World/Other', 'german': 'World/Other',
    'swedish': 'World/Other', 'british': 'World/Other',
    'j-rock': 'World/Other', 'anime': 'World/Other',
    'children': 'World/Other', 'kids': 'World/Other',
    'comedy': 'World/Other', 'sad': 'World/Other',
    'chill': 'World/Other', 'dub': 'World/Other',
}

In [19]:
df  = pd.read_csv('/Users/Owner/Desktop/DSCourses/DS 340/340_Project/DS-340-Group-Project/Data/spotify-tracks-dataset.csv')


In [ ]:
# sampling math

MIN_SAMPLES_PER_GENRE = 2000   # 50000 / ~25 genres ≈ 2000 floor per genre
SAMPLES_PER_SUBGENRE  = 439

PERSON_A_GENRES = {'Rock', 'Metal', 'Electronic', 'House/Dance', 'Pop', 'Hip-Hop/R&B'}
PERSON_B_GENRES = {'Latin', 'Jazz/Blues', 'Classical/Instrumental', 'Country/Folk', 'Reggae', 'World/Other'}

PERSON_A_BASE_PER_SUBGENRE = 439   # 63 subgenres × 439 ≈ 27,657 tracks
PERSON_B_BASE_PER_SUBGENRE = 439   # 51 subgenres × 439 ≈ 22,389 tracks

In [ ]:
# Map subgenres to parent genres FIRST --> was able to get the sampled person dataset

#df['genre'] = df['track_genre'].map(GENRE_MAP)
#df = df.dropna(subset=['genre'])


#df_a = run_sampling(df, PERSON_A_GENRES, PERSON_A_BASE_PER_SUBGENRE, 'sampled_person_a.csv')
#df_b = run_sampling(df, PERSON_B_GENRES, PERSON_B_BASE_PER_SUBGENRE, 'sampled_person_b.csv')


Saved 24702 tracks → sampled_person_a.csv
genre
Electronic     5392
Hip-Hop/R&B    1857
House/Dance    3428
Metal          3579
Pop            6042
Rock           4404
[WARN] bossanova (Latin): requested 439, only 0 available

Saved 21566 tracks → sampled_person_b.csv
genre
Classical/Instrumental    3018
Country/Folk              2666
Jazz/Blues                1928
Latin                     5110
Reggae                    1337
World/Other               7507


In [22]:
#helper functions for sampling


 
def build_genre_to_subgenres(genre_map: dict) -> dict:
    """Invert GENRE_MAP: genre label -> list of subgenre keys."""
    genre_to_subs = {}
    for subgenre, genre in genre_map.items():
        genre_to_subs.setdefault(genre, []).append(subgenre)
    return genre_to_subs
 
 
def compute_sampling_plan(
    genre_to_subs: dict,
    min_per_genre: int = MIN_SAMPLES_PER_GENRE,
    base_per_subgenre: int = SAMPLES_PER_SUBGENRE,
) -> dict:
    """
    Returns a plan: { genre: { subgenre: n_samples } }
 
    Logic:
      - Natural allocation = base_per_subgenre * n_subgenres
      - If natural allocation < min_per_genre, scale up evenly so the
        total hits the floor (distributed as evenly as possible).
    """
    plan = {}
    for genre, subgenres in genre_to_subs.items():
        n_subs = len(subgenres)
        natural_total = base_per_subgenre * n_subs
 
        if natural_total >= min_per_genre:
            per_sub, remainder = base_per_subgenre, 0
        else:
            per_sub, remainder = divmod(min_per_genre, n_subs)
 
        plan[genre] = {
            sub: per_sub + (1 if i < remainder else 0)
            for i, sub in enumerate(subgenres)
        }
    return plan
 
 
def print_plan_summary(plan: dict) -> None:
    print(f"\n{'Genre':<25} {'Subgenres':>9} {'Total':>7}")
    print("-" * 44)
    grand_total = 0
    for genre, alloc in sorted(plan.items()):
        total = sum(alloc.values())
        grand_total += total
        print(f"{genre:<25} {len(alloc):>9} {total:>7}")
    print("-" * 44)
    print(f"{'GRAND TOTAL':<25} {'':>9} {grand_total:>7}\n")
 
 
def run_sampling(
    df: pd.DataFrame,
    assigned_genres: set,
    base_per_subgenre: int,
    output_file: str,
    random_state: int = 42,
) -> pd.DataFrame:
    """Sample tracks for one person's assigned genres and save to CSV."""
    genre_map_subset = {k: v for k, v in GENRE_MAP.items() if v in assigned_genres}
    genre_to_subs    = build_genre_to_subgenres(genre_map_subset)
    plan             = compute_sampling_plan(genre_to_subs, MIN_SAMPLES_PER_GENRE, base_per_subgenre)
 
    sampled_frames = []
    for genre, subgenre_counts in plan.items():
        for subgenre, n in subgenre_counts.items():
            subset = df[df['track_genre'] == subgenre]
            k = min(n, len(subset))
            if k < n:
                print(f"[WARN] {subgenre} ({genre}): requested {n}, only {k} available")
            if k > 0:
                sampled_frames.append(subset.sample(k, random_state=random_state))
 
    result = (
        pd.concat(sampled_frames)
        .drop_duplicates(subset='track_id')
        .reset_index(drop=True)
    )
    result.to_csv(output_file, index=False)
    print(f"\nSaved {len(result)} tracks → {output_file}")
    print(result.groupby('genre')['track_id'].count().to_string())
    return result

In [8]:
#audio heler functions

def _find_output(out_path: str) -> Optional[str]:
    """Find the actual downloaded file regardless of extension."""
    matches = glob.glob(out_path + '.*')
    wav = [f for f in matches if f.endswith('.wav')]
    return wav[0] if wav else (matches[0] if matches else None)
 
 
def get_audio_clip(
    track_name: str,
    artist: str,
    duration_ms: int = None,
    out_dir: str = 'audio_clips',
) -> Optional[str]:
    os.makedirs(out_dir, exist_ok=True)
 
    query     = f"{track_name} {artist} official audio"
    safe_name = f"{track_name[:40]}_{artist[:20]}".replace('/', '_').replace(' ', '_')
    out_path  = os.path.join(out_dir, safe_name)
 
    # Skip if already downloaded
    existing = _find_output(out_path)
    if existing:
        return existing
 
    if duration_ms:
        mid   = (duration_ms / 1000) // 2
        start = int(mid - 15)
        end   = int(mid + 15)
    else:
        start, end = 60, 90
 
    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': out_path + '.%(ext)s',
        'quiet': True,
        'no_warnings': True,
        'postprocessors': [{'key': 'FFmpegExtractAudio', 'preferredcodec': 'wav'}],
        'download_sections': [{'start_time': start, 'end_time': end}],
        'default_search': 'ytsearch1',
    }
 
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([query])
        return _find_output(out_path)
    except Exception as e:
        print(f"[ERROR] {track_name} - {artist}: {e}")
        return None

In [9]:
#feature extraction

def extract_features(filepath: str, n_mfcc: int = 13) -> Optional[np.ndarray]:
    try:
        audio, sr = librosa.load(filepath, sr=22050, duration=30)
 
        mfcc     = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)
        delta    = librosa.feature.delta(mfcc)
        chroma   = librosa.feature.chroma_stft(y=audio, sr=sr)
        contrast = librosa.feature.spectral_contrast(y=audio, sr=sr)
        zcr      = librosa.feature.zero_crossing_rate(y=audio)
 
        return np.concatenate([
            mfcc.mean(axis=1),      # 13
            mfcc.std(axis=1),       # 13
            delta.mean(axis=1),     # 13
            chroma.mean(axis=1),    # 12
            contrast.mean(axis=1),  # 7
            zcr.mean(axis=1),       # 1
        ])  # → 59-dim vector
 
    except Exception as e:
        print(f"[ERROR] Feature extraction failed for {filepath}: {e}")
        return None
 

In [ ]:
#mfcc dataset builder

def build_mfcc_dataset(
    df: pd.DataFrame,
    n_mfcc: int = 13,
    checkpoint_file: str = 'mfcc_features_checkpoint.csv',
    sleep_between: float = 0.5,
) -> pd.DataFrame:
 
    # Resume from checkpoint if it exists
    if os.path.exists(checkpoint_file):
        existing = pd.read_csv(checkpoint_file)
        done_ids = set(existing['track_id'])
        records  = existing.to_dict('records')
        df       = df[~df['track_id'].isin(done_ids)].reset_index(drop=True)
        print(f"Resuming: {len(done_ids)} already done, {len(df)} remaining.")
    else:
        records, done_ids = [], set()
 
    feature_cols = None
    total        = len(df)
 
    for i, (_, row) in enumerate(df.iterrows()):
        print(f"[{i+1}/{total}] {row['track_name']} — {row['artists']}")
 
        filepath = get_audio_clip(
            row['track_name'],
            row['artists'],
            duration_ms=row.get('duration_ms'),
        )
 
        if not filepath or not os.path.exists(filepath):
            print("  ↳ skipped (no audio)")
            continue
 
        features = extract_features(filepath, n_mfcc=n_mfcc)
 
        if features is None:
            os.remove(filepath)
            print("  ↳ skipped (feature extraction failed)")
            continue
 
        os.remove(filepath)
 
        # Derive column names from first successful extraction
        if feature_cols is None:
            feature_cols = [f'feat_{j}' for j in range(len(features))]
 
        records.append({
            'track_id':    row['track_id'],
            'track_genre': row['track_genre'],
            'genre':       row['genre'],
            **dict(zip(feature_cols, features))
        })
 
        # Checkpoint every 50 tracks
        if len(records) % 50 == 0:
            pd.DataFrame(records).to_csv(checkpoint_file, index=False)
            print(f"  ✓ checkpoint saved ({len(records)} tracks so far)")
 
        time.sleep(sleep_between)  # avoid YT rate limiting
 
    mfcc_df = pd.DataFrame(records)
    mfcc_df.to_csv('mfcc_features.csv', index=False)
    print(f"\nDone! Extracted features for {len(records)}/{total + len(done_ids)} tracks.")
    return mfcc_df
 

In [ ]:
#mfcc Running 

    # 2. Sample tracks per person
print("=== Person A ===")
df_a = run_sampling(df, PERSON_A_GENRES, PERSON_A_BASE_PER_SUBGENRE, 'sampled_person_a.csv')
 
print("\n=== Person B ===")
df_b = run_sampling(df, PERSON_B_GENRES, PERSON_B_BASE_PER_SUBGENRE, 'sampled_person_b.csv')
 
    # 3. Each person runs this on their own sampled CSV:
    #
    #   Person A(cindy) runs:
    #     sampled = pd.read_csv('sampled_person_a.csv')
    #     mfcc_df = build_mfcc_dataset(sampled, checkpoint_file='checkpoint_a.csv')
    #
    #   Person B(aymen)runs:
    #     sampled = pd.read_csv('sampled_person_b.csv')
    #     mfcc_df = build_mfcc_dataset(sampled, checkpoint_file='checkpoint_b.csv')
    #
    # 4. Merge both outputs at the end:
    #     final = pd.concat([
    #         pd.read_csv('mfcc_features_a.csv'),
    #         pd.read_csv('mfcc_features_b.csv'),
    #     ]).reset_index(drop=True)
    #     final.to_csv('mfcc_features_final.csv', index=False)
    

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,...,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre,genre
0,2Wzl4bZLjqFUOeZaGrTaVJ,YUNGBLUD,Drippy Drippy,parents,0,172026,True,0.594,0.833,9,...,0,0.0507,0.00967,0.000000,0.300,0.599,82.040,4,rock,Rock
1,1UEj7RpdqH02grDvdxTKLP,All Time Low,Alternative Christmas 2022,"Merry Christmas, Kiss My Ass",0,199253,False,0.574,0.966,4,...,1,0.0685,0.01410,0.000000,0.430,0.788,128.034,4,rock,Rock
2,3SCYKwSFwlSe5xEvVlcP1W,George Strait,Country Christmas Greatest Hits,Jingle Bell Rock,1,132493,False,0.775,0.562,4,...,0,0.0299,0.46800,0.000000,0.174,0.770,121.034,4,rock,Rock
3,1NhPKVLsHhFUHIOZ32QnS2,OneRepublic,Waking Up,Secrets,76,224693,False,0.516,0.764,2,...,1,0.0366,0.07170,0.000000,0.115,0.376,148.021,4,rock,Rock
4,1ihSSnA4d0dcxQouy7gtNJ,Juanes,Retro Baladas,Es Por Ti,0,251120,False,0.694,0.761,4,...,1,0.0263,0.17600,0.000036,0.123,0.866,129.953,4,rock,Rock


In [16]:
# Using YT to extract sngs

from typing import Optional

def get_audio_clip(track_name: str, artist: str, duration_ms: int = None, out_dir: str = 'audio_clips') -> Optional[str]:
    os.makedirs(out_dir, exist_ok=True)
    query = f"{track_name} {artist} official audio"
    safe_name = f"{track_name[:40]}_{artist[:20]}".replace('/', '_').replace(' ', '_')
    out_path = os.path.join(out_dir, safe_name)

    if duration_ms:
        mid = (duration_ms / 1000) // 2
        start = int(mid - 15)
        end   = int(mid + 15)
    else:
        start, end = 60, 90

    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': out_path + '.%(ext)s',
        'quiet': True,
        'no_warnings': True,
        'postprocessors': [{'key': 'FFmpegExtractAudio', 'preferredcodec': 'wav'}],
        'download_sections': {f'*{start}-{end}': True},
        'default_search': 'ytsearch1',
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([query])
        return out_path + '.wav'
    except Exception as e:
        print(f"[ERROR] {track_name} - {artist}: {e}")
        return None


def extract_mfcc(filepath: str, n_mfcc: int = 13) -> Optional[np.ndarray]:
    try:
        audio, sr = librosa.load(filepath, sr=22050, duration=30)
        mfcc      = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)
        delta     = librosa.feature.delta(mfcc)
        return np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1), delta.mean(axis=1)])
    except Exception as e:
        print(f"[ERROR] MFCC extraction failed for {filepath}: {e}")
        return None



In [17]:
#test
test_row = sampled_df.iloc[0]
print(f"Testing: {test_row['track_name']} — {test_row['artists']}")

filepath = get_audio_clip(test_row['track_name'], test_row['artists'], duration_ms=test_row['duration_ms'])
print(f"Audio saved to: {filepath}")

features = extract_mfcc(filepath)
print(f"MFCC shape: {features.shape}")
print(f"First few values: {features[:5]}")

Testing: parents — YUNGBLUD
Audio saved to: audio_clips/parents_YUNGBLUD.wav           
MFCC shape: (39,)
First few values: [-45.296093   66.90737     7.7488155  14.39976     5.574636 ]


In [19]:
def build_mfcc_dataset(df: pd.DataFrame, n_mfcc: int = 13) -> pd.DataFrame:
    records = []
    feature_cols = (
        [f'mfcc_mean_{i+1}'  for i in range(n_mfcc)] +
        [f'mfcc_std_{i+1}'   for i in range(n_mfcc)] +
        [f'mfcc_delta_{i+1}' for i in range(n_mfcc)]
    )

    total = len(df)
    for i, (_, row) in enumerate(df.iterrows()):
        print(f"[{i+1}/{total}] {row['track_name']} — {row['artists']}")

        filepath = get_audio_clip(row['track_name'], row['artists'], duration_ms=row['duration_ms'])
        if not filepath or not os.path.exists(filepath):
            print("  ↳ skipped (no audio)")
            continue

        features = extract_mfcc(filepath)
        os.remove(filepath)

        if features is None:
            print("  ↳ skipped (MFCC failed)")
            continue

        records.append({
            'track_id':    row['track_id'],
            'track_genre': row['track_genre'],
            'genre':       row['genre'],
            **dict(zip(feature_cols, features))
        })

        # Checkpoint every 50 tracks
        if len(records) % 50 == 0:
            pd.DataFrame(records).to_csv('mfcc_features_checkpoint.csv', index=False)
            print(f"  ✓ checkpoint saved ({len(records)} tracks so far)")

    mfcc_df = pd.DataFrame(records)
    mfcc_df.to_csv('mfcc_features.csv', index=False)
    print(f"\nDone! Extracted MFCCs for {len(records)}/{total} tracks.")
    return mfcc_df

In [21]:
# running for full mfcc
mfcc_df = build_mfcc_dataset(sampled_df)


[1/1164] parents — YUNGBLUD
[2/1164] Merry Christmas, Kiss My Ass — All Time Low     
[3/1164] Jingle Bell Rock — George Strait                
[4/1164] Secrets — OneRepublic                           
[5/1164] Es Por Ti — Juanes                              
[6/1164] Winter Wonderland — Rod Stewart;Michael Bublé   
[7/1164] If I Had You — Rod Stewart                      
[8/1164] Thunderstruck — AC/DC                             
[9/1164] I Belong To You — Lenny Kravitz                   
[10/1164] Black Summer — Red Hot Chili Peppers           
[11/1164] Finish Line — Skillet                          
[12/1164] Won't Back Down — Fuel                         
[13/1164] Ciencia De La Lluvia — Enjambre                
[14/1164] Living Hope — Phil Wickham                     
[15/1164] Middle Finger — Bohnes                           


ERROR: [youtube] ohocp8JWbGQ: This video is not available


[ERROR] Middle Finger - Bohnes: ERROR: [youtube] ohocp8JWbGQ: This video is not available
  ↳ skipped (no audio)
[16/1164] El Recuento de los Daños — Los Estramboticos
[17/1164] In The Name Of Love - Recorded At Spotify Studios NYC — Kari Jobe
[18/1164] Awake and Alive — Skillet                      
[19/1164] Playground (from the series Arcane League of Legends) — Bea Miller
[20/1164] Symphony Of Destruction — Megadeth             
[21/1164] Punk Superstars — GUFI                           
[22/1164] Heute hier, morgen dort - Live — Die Toten Hosen
[23/1164] Pretty Fly (For A White Guy) — The Offspring   
[24/1164] Beat the Devil's Tattoo — Black Rebel Motorcycle Club
[25/1164] Adela en el Carrousell — Charly García         
[26/1164] Amnesia — Say Ocean                            
[27/1164] Você Sabe — Autoramas                           
[28/1164] 80's — Allison                                 
[29/1164] Always — blink-182                             
[30/1164] Spellbound — Siouxsi

ERROR: [youtube] iex6eui5o0s: Sign in to confirm your age. This video may be inappropriate for some users. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


[ERROR] Fat Fuck - CKY: ERROR: [youtube] iex6eui5o0s: Sign in to confirm your age. This video may be inappropriate for some users. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies
  ↳ skipped (no audio)
[197/1164] Fatal Fall — Flaw
[198/1164] Snap Your Fingers, Snap Your Neck — Prong       
[199/1164] Bender — Sevendust                            
[200/1164] Electric Avenue — Powerman 5000               
[201/1164] Tie Me Down (with Elley Duhé) — Gryffin;Elley Duhé
[202/1164] Hollywood Perfect — Unknown Brain;NotEvenTanner 
[203/1164] The Bender — Matoma;Brando                      
[204/1164] Poison — Rita Ora                               
  ✓ checkpoint saved (200 tracks so far)                   
[205/1164] Takeaway (feat

...

[ERROR] Wake Up - Christopher Zondaflex Tyler: ERROR: [youtube] Yo1P0u_1xes: This video is not available
  ↳ skipped (no audio)
[1132/1164] One Two Buckle My Shoe — Kidsounds


ERROR: [youtube] eT4BwDl7yHI: This video is not available


[ERROR] One Two Buckle My Shoe - Kidsounds: ERROR: [youtube] eT4BwDl7yHI: This video is not available
  ↳ skipped (no audio)
[1133/1164] Brush Your Teeth — Desmond Dennis
[1134/1164] Daddy Minion — Desmond Dennis                


ERROR: [youtube] OLV02G_83YI: This video is not available


[ERROR] Daddy Minion - Desmond Dennis: ERROR: [youtube] OLV02G_83YI: This video is not available
  ↳ skipped (no audio)
[1135/1164] I Killed a Baby — Christopher Titus
[1136/1164] Country Music / Mexican Cowboy / Tough Cowboy — Pablo Francisco
[1137/1164] The Spirit Cave — Patton Oswalt              
[1138/1164] Snootch — Iliza Shlesinger                   
[1139/1164] A Basket of Kisses — Greg Proops               
[1140/1164] Oscar Movies and Paula Deen — D.L. Hughley     
[1141/1164] New York to LA — Andy Hendrickson              
[1142/1164] My Rehab Counselor — Artie Lange             
[1143/1164] Brittni and Judah — Lil Rel Howery           
  ↳ skipped (no audio)
[1144/1164] Wedding Story — Mike Birbiglia
[1145/1164] I fucked up — convolk                          
[1146/1164] the overthinker — One Hope                   
[1147/1164] circus — PmBata                              
[1148/1164] used to — Josh Golden                        
[1149/1164] Ease Off — YNG Martyr           